In [ ]:
pip install tensorboardX

In [ ]:
%%writefile util.py

# Paste your complete util.py code below
import sys
import os
import torch


def mkdirs(paths):
    if isinstance(paths, list) and not isinstance(paths, str):
        for path in paths:
            mkdir(path)
    else:
        mkdir(paths)


def mkdir(path):
    if not os.path.exists(path):
        os.makedirs(path)


def unnormalize(tens, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    # assume tensor of shape NxCxHxW
    return tens * torch.Tensor(std)[None, :, None, None] + torch.Tensor(
        mean)[None, :, None, None]




class Logger(object):
    """Log stdout messages."""

    def __init__(self, outfile):
        self.terminal = sys.stdout
        self.log = open(outfile, "a")
        sys.stdout = self

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()


def printSet(set_str):
    set_str = str(set_str)
    num = len(set_str)
    print("="*num*3)
    print(" "*num + set_str)
    print("="*num*3)

Writing util.py


In [ ]:
%%writefile datasets.py

# Paste your complete util.py code below
import cv2
import numpy as np
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from random import random, choice
from io import BytesIO
from PIL import Image
from PIL import ImageFile
from scipy.ndimage.filters import gaussian_filter
from torchvision.transforms import InterpolationMode

ImageFile.LOAD_TRUNCATED_IMAGES = True

def dataset_folder(opt, root):
    if opt.mode == 'binary':
        return binary_dataset(opt, root)
    if opt.mode == 'filename':
        return FileNameDataset(opt, root)
    raise ValueError('opt.mode needs to be binary or filename.')


def binary_dataset(opt, root):
    if opt.isTrain:
        crop_func = transforms.RandomCrop(opt.cropSize)
    elif opt.no_crop:
        crop_func = transforms.Lambda(lambda img: img)
    else:
        crop_func = transforms.CenterCrop(opt.cropSize)

    if opt.isTrain and not opt.no_flip:
        flip_func = transforms.RandomHorizontalFlip()
    else:
        flip_func = transforms.Lambda(lambda img: img)
    if not opt.isTrain and opt.no_resize:
        rz_func = transforms.Lambda(lambda img: img)
    else:
        # rz_func = transforms.Lambda(lambda img: custom_resize(img, opt))
        rz_func = transforms.Resize((opt.loadSize, opt.loadSize))

    dset = datasets.ImageFolder(
            root,
            transforms.Compose([
                rz_func,
                # transforms.Lambda(lambda img: data_augment(img, opt)),
                crop_func,
                flip_func,
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]))
    return dset


class FileNameDataset(datasets.ImageFolder):
    def name(self):
        return 'FileNameDataset'

    def __init__(self, opt, root):
        self.opt = opt
        super().__init__(root)

    def __getitem__(self, index):
        # Loading sample
        path, target = self.samples[index]
        return path


def data_augment(img, opt):
    img = np.array(img)

    if random() < opt.blur_prob:
        sig = sample_continuous(opt.blur_sig)
        gaussian_blur(img, sig)

    if random() < opt.jpg_prob:
        method = sample_discrete(opt.jpg_method)
        qual = sample_discrete(opt.jpg_qual)
        img = jpeg_from_key(img, qual, method)

    return Image.fromarray(img)


def sample_continuous(s):
    if len(s) == 1:
        return s[0]
    if len(s) == 2:
        rg = s[1] - s[0]
        return random() * rg + s[0]
    raise ValueError("Length of iterable s should be 1 or 2.")


def sample_discrete(s):
    if len(s) == 1:
        return s[0]
    return choice(s)


def gaussian_blur(img, sigma):
    gaussian_filter(img[:,:,0], output=img[:,:,0], sigma=sigma)
    gaussian_filter(img[:,:,1], output=img[:,:,1], sigma=sigma)
    gaussian_filter(img[:,:,2], output=img[:,:,2], sigma=sigma)


def cv2_jpg(img, compress_val):
    img_cv2 = img[:,:,::-1]
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), compress_val]
    result, encimg = cv2.imencode('.jpg', img_cv2, encode_param)
    decimg = cv2.imdecode(encimg, 1)
    return decimg[:,:,::-1]


def pil_jpg(img, compress_val):
    out = BytesIO()
    img = Image.fromarray(img)
    img.save(out, format='jpeg', quality=compress_val)
    img = Image.open(out)
    # load from memory before ByteIO closes
    img = np.array(img)
    out.close()
    return img


jpeg_dict = {'cv2': cv2_jpg, 'pil': pil_jpg}
def jpeg_from_key(img, compress_val, key):
    method = jpeg_dict[key]
    return method(img, compress_val)


# rz_dict = {'bilinear': Image.BILINEAR,
           # 'bicubic': Image.BICUBIC,
           # 'lanczos': Image.LANCZOS,
           # 'nearest': Image.NEAREST}
rz_dict = {'bilinear': InterpolationMode.BILINEAR,
           'bicubic': InterpolationMode.BICUBIC,
           'lanczos': InterpolationMode.LANCZOS,
           'nearest': InterpolationMode.NEAREST}
def custom_resize(img, opt):
    interp = sample_discrete(opt.rz_interp)
    return TF.resize(img, (opt.loadSize,opt.loadSize), interpolation=rz_dict[interp])

Writing datasets.py


In [ ]:
%%writefile init.py
import torch
import numpy as np
from torch.utils.data.sampler import WeightedRandomSampler

from datasets import dataset_folder

'''
def get_dataset(opt):
    dset_lst = []
    for cls in opt.classes:
        root = opt.dataroot + '/' + cls
        dset = dataset_folder(opt, root)
        dset_lst.append(dset)
    return torch.utils.data.ConcatDataset(dset_lst)
'''

import os
def get_dataset(opt):
    classes = os.listdir(opt.dataroot) if len(opt.classes) == 0 else opt.classes
    if '0_real' not in classes or '1_fake' not in classes:
        dset_lst = []
        for cls in classes:
            root = opt.dataroot + '/' + cls
            dset = dataset_folder(opt, root)
            dset_lst.append(dset)
        return torch.utils.data.ConcatDataset(dset_lst)
    return dataset_folder(opt, opt.dataroot)

def get_bal_sampler(dataset):
    targets = []
    for d in dataset.datasets:
        targets.extend(d.targets)

    ratio = np.bincount(targets)
    w = 1. / torch.tensor(ratio, dtype=torch.float)
    sample_weights = w[targets]
    sampler = WeightedRandomSampler(weights=sample_weights,
                                    num_samples=len(sample_weights))
    return sampler


def create_dataloader(opt):
    shuffle = not opt.serial_batches if (opt.isTrain and not opt.class_bal) else False
    dataset = get_dataset(opt)
    sampler = get_bal_sampler(dataset) if opt.class_bal else None

    data_loader = torch.utils.data.DataLoader(dataset,
                                              batch_size=opt.batch_size,
                                              shuffle=shuffle,
                                              sampler=sampler,
                                              num_workers=int(opt.num_threads))
    return data_loader

Writing init.py


In [ ]:

%%writefile base_options.py

# Paste your complete util.py code below
# Paste your complete util.py code below

# Paste the complete validate.py code here
import argparse
import os
import time
import util
import torch
#import models
#import data


class BaseOptions():
    def __init__(self):
        self.initialized = False

    def initialize(self, parser):
        parser.add_argument('--mode', default='binary')
        parser.add_argument('--arch', type=str, default='res50', help='architecture for binary classification')

        # data augmentation
        parser.add_argument('--rz_interp', default='bilinear')
        parser.add_argument('--blur_prob', type=float, default=0)
        parser.add_argument('--blur_sig', default='0.5')
        parser.add_argument('--jpg_prob', type=float, default=0)
        parser.add_argument('--jpg_method', default='cv2')
        parser.add_argument('--jpg_qual', default='75')

        parser.add_argument('--dataroot', default='./dataset/', help='path to images (should have subfolders trainA, trainB, valA, valB, etc)')
        parser.add_argument('--classes', default='', help='image classes to train on')
        parser.add_argument('--class_bal', action='store_true')
        parser.add_argument('--batch_size', type=int, default=64, help='input batch size')
        parser.add_argument('--loadSize', type=int, default=256, help='scale images to this size')
        parser.add_argument('--cropSize', type=int, default=224, help='then crop to this size')
        parser.add_argument('--gpu_ids', type=str, default='0', help='gpu ids: e.g. 0  0,1,2, 0,2. use -1 for CPU')
        parser.add_argument('--name', type=str, default='experiment_name', help='name of the experiment. It decides where to store samples and models')
        parser.add_argument('--epoch', type=str, default='latest', help='which epoch to load? set to latest to use latest cached model')
        parser.add_argument('--num_threads', default=8, type=int, help='# threads for loading data')
        parser.add_argument('--checkpoints_dir', type=str, default='./checkpoints', help='models are saved here')
        parser.add_argument('--serial_batches', action='store_true', help='if true, takes images in order to make batches, otherwise takes them randomly')
        parser.add_argument('--resize_or_crop', type=str, default='scale_and_crop', help='scaling and cropping of images at load time [resize_and_crop|crop|scale_width|scale_width_and_crop|none]')
        parser.add_argument('--no_flip', action='store_true', help='if specified, do not flip the images for data augmentation')
        parser.add_argument('--init_type', type=str, default='normal', help='network initialization [normal|xavier|kaiming|orthogonal]')
        parser.add_argument('--init_gain', type=float, default=0.02, help='scaling factor for normal, xavier and orthogonal.')
        parser.add_argument('--suffix', default='', type=str, help='customized suffix: opt.name = opt.name + suffix: e.g., {model}_{netG}_size{loadSize}')
        parser.add_argument('--delr_freq', type=int, default=20, help='frequency of changing lr')


        self.initialized = True
        return parser

    def gather_options(self):
        # initialize parser with basic options
        if not self.initialized:
            parser = argparse.ArgumentParser(
                formatter_class=argparse.ArgumentDefaultsHelpFormatter)
            parser = self.initialize(parser)

        # get the basic options
        opt, _ = parser.parse_known_args()
        self.parser = parser

        return opt #parser.parse_args()

    def print_options(self, opt):
        message = ''
        message += '----------------- Options ---------------\n'
        for k, v in sorted(vars(opt).items()):
            comment = ''
            default = self.parser.get_default(k)
            if v != default:
                comment = '\t[default: %s]' % str(default)
            message += '{:>25}: {:<30}{}\n'.format(str(k), str(v), comment)
        message += '----------------- End -------------------'
        print(message)

        # save to the disk

        expr_dir = os.path.join(opt.checkpoints_dir, opt.name)
        util.mkdirs(expr_dir)
        file_name = os.path.join(expr_dir, 'opt.txt')
        with open(file_name, 'wt') as opt_file:
            opt_file.write(message)
            opt_file.write('\n')

    def parse(self, print_options=True):

        opt = self.gather_options()
        opt.isTrain = self.isTrain   # train or test
        opt.name = opt.name + time.strftime("%Y_%m_%d_%H_%M_%S", time.localtime())
        # process opt.suffix
        if opt.suffix:
            suffix = ('_' + opt.suffix.format(**vars(opt))) if opt.suffix != '' else ''
            opt.name = opt.name + suffix

        if print_options:
            self.print_options(opt)

        # set gpu ids
        str_ids = opt.gpu_ids.split(',')
        opt.gpu_ids = []
        for str_id in str_ids:
            id = int(str_id)
            if id >= 0:
                opt.gpu_ids.append(id)
        if len(opt.gpu_ids) > 0:
            torch.cuda.set_device(opt.gpu_ids[0])

        # additional
        opt.classes = opt.classes.split(',')
        opt.rz_interp = opt.rz_interp.split(',')
        opt.blur_sig = [float(s) for s in opt.blur_sig.split(',')]
        opt.jpg_method = opt.jpg_method.split(',')
        opt.jpg_qual = [int(s) for s in opt.jpg_qual.split(',')]
        if len(opt.jpg_qual) == 2:
            opt.jpg_qual = list(range(opt.jpg_qual[0], opt.jpg_qual[1] + 1))
        elif len(opt.jpg_qual) > 2:
            raise ValueError("Shouldn't have more than 2 values for --jpg_qual.")

        self.opt = opt
        return self.opt

Writing base_options.py


In [ ]:
%%writefile train_options.py

# Paste your complete util.py code below

# Paste your complete util.py code below

# Paste your complete util.py code below
from base_options import BaseOptions


class TrainOptions(BaseOptions):
    def initialize(self, parser):
        parser = BaseOptions.initialize(self, parser)
        parser.add_argument('--earlystop_epoch', type=int, default=15)
        parser.add_argument('--data_aug', action='store_true', help='if specified, perform additional data augmentation (photometric, blurring, jpegging)')
        parser.add_argument('--optim', type=str, default='adam', help='optim to use [sgd, adam]')
        parser.add_argument('--new_optim', action='store_true', help='new optimizer instead of loading the optim state')
        parser.add_argument('--loss_freq', type=int, default=400, help='frequency of showing loss on tensorboard')
        parser.add_argument('--save_latest_freq', type=int, default=2000, help='frequency of saving the latest results')
        parser.add_argument('--save_epoch_freq', type=int, default=20, help='frequency of saving checkpoints at the end of epochs')
        parser.add_argument('--continue_train', action='store_true', help='continue training: load the latest model')
        parser.add_argument('--epoch_count', type=int, default=1, help='the starting epoch count, we save the model by <epoch_count>, <epoch_count>+<save_latest_freq>, ...')
        parser.add_argument('--last_epoch', type=int, default=-1, help='starting epoch count for scheduler intialization')
        parser.add_argument('--train_split', type=str, default='train', help='train, val, test, etc')
        parser.add_argument('--val_split', type=str, default='val', help='train, val, test, etc')
        parser.add_argument('--niter', type=int, default=1000, help='# of iter at starting learning rate')
        parser.add_argument('--beta1', type=float, default=0.9, help='momentum term of adam')
        parser.add_argument('--lr', type=float, default=0.0001, help='initial learning rate for adam')
        # parser.add_argument('--model_path')
        # parser.add_argument('--no_resize', action='store_true')
        # parser.add_argument('--no_crop', action='store_true')
        self.isTrain = True
        return parser

Writing train_options.py


In [ ]:
%%writefile freqnet.py
import torch.nn as nn
from torch.nn import functional as F
import torch

__all__ = ['FreqNet, freqnet']


def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = conv1x1(inplanes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class FreqNet(nn.Module):

    def __init__(self, block=Bottleneck, layers=[3, 4], num_classes=1, zero_init_residual=False):
        super(FreqNet, self).__init__()



        self.weight1 = nn.Parameter(torch.randn((64, 3, 1, 1)).cuda())
        self.bias1   = nn.Parameter(torch.randn((64,)).cuda())
        self.realconv1 = conv1x1(64, 64, stride=1)
        self.imagconv1 = conv1x1(64, 64, stride=1)

        self.weight2 = nn.Parameter(torch.randn((64, 64, 1, 1)).cuda())
        self.bias2   = nn.Parameter(torch.randn((64,)).cuda())
        self.realconv2 = conv1x1(64, 64, stride=1)
        self.imagconv2 = conv1x1(64, 64, stride=1)


        self.weight3 = nn.Parameter(torch.randn((256, 256, 1, 1)).cuda())
        self.bias3   = nn.Parameter(torch.randn((256,)).cuda())
        self.realconv3 = conv1x1(256, 256, stride=1)
        self.imagconv3 = conv1x1(256, 256, stride=1)

        self.weight4 = nn.Parameter(torch.randn((256, 256, 1, 1)).cuda())
        self.bias4   = nn.Parameter(torch.randn((256,)).cuda())
        self.realconv4 = conv1x1(256, 256, stride=1)
        self.imagconv4 = conv1x1(256, 256, stride=1)

        self.inplanes = 64
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])            # layer1 of resnet50
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2) # layer2 of resnet50
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 1)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def hfreqWH(self, x, scale):
     assert scale > 2

     # 2D Fourier transform
     x = torch.fft.fft2(x, norm="ortho")
     x = torch.fft.fftshift(x, dim=[-2, -1])

     b, c, h, w = x.shape

     # Frequency coordinates centered at zero
     yy = torch.arange(h, device=x.device, dtype=torch.float32) - h // 2
     xx = torch.arange(w, device=x.device, dtype=torch.float32) - w // 2

     # Normalize frequency coordinates
     yy = yy / max(h / scale, 1.0)
     xx = xx / max(w / scale, 1.0)

     Y, X = torch.meshgrid(yy, xx, indexing='ij')

     # Radial distance
     radius = torch.sqrt(Y**2 + X**2)

     # Smooth radial high-pass filter
     transition_width = 0.1

     mask = torch.sigmoid(
        (radius - 1.0) / transition_width
     )

     # Apply mask
     x = x * mask.unsqueeze(0).unsqueeze(0)

     # Inverse Fourier transform
     x = torch.fft.ifftshift(x, dim=[-2, -1])
     x = torch.fft.ifft2(x, norm="ortho")

     x = torch.real(x)
     x = F.relu(x, inplace=True)

     return x

    def hfreqC(self, x, scale):
        assert scale>2
        # print(f'input shape: {x.shape}, min: {x.min()}, max: {x.max()}, mean: {x.mean()}')
        x = torch.fft.fft(x, dim=1, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=1)
        b,c,h,w = x.shape

        x[:,c//2-c//scale:c//2+c//scale,:,:] = 0.0

        x = torch.fft.ifftshift(x, dim=1)
        x = torch.fft.ifft(x, dim=1, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)

        # print(f'output shape: {x.shape}, min: {x.min()}, max: {x.max()}, mean: {x.mean()}')
        # print()
        return x


    def forward(self, x):

        ### HFRI
        x = self.hfreqWH(x, 4)
        x = F.conv2d(x, self.weight1, self.bias1, stride=1, padding=0)
        x = F.relu(x, inplace=True)

        ### HFRFC
        x = self.hfreqC(x, 4)

        ### FCL
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(self.realconv1(x.real), self.imagconv1(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)

        ### HFRFS
        x = self.hfreqWH(x, 4)
        x = F.conv2d(x, self.weight2, self.bias2, stride=2, padding=0)
        x = F.relu(x, inplace=True)

        ### HFRFC
        x = self.hfreqC(x, 4)

        ### FCL
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(self.realconv2(x.real), self.imagconv2(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)


        x = self.maxpool(x)
        x = self.layer1(x)# in64 out256


        ### HFRFS
        x = self.hfreqWH(x, 4)
        x = F.conv2d(x, self.weight3, self.bias3, stride=1, padding=0)
        x = F.relu(x, inplace=True)


        ### FCL
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(self.realconv3(x.real), self.imagconv3(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)

        ### HFRFS
        x = self.hfreqWH(x, 4)
        x = F.conv2d(x, self.weight4, self.bias4, stride=2, padding=0)
        x = F.relu(x, inplace=True)

        ### FCL
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(self.realconv4(x.real), self.imagconv4(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)

        x = self.layer2(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)

        return x

def freqnet(**kwargs):

    return FreqNet()

Writing freqnet.py


In [ ]:
%%writefile freqnet1.py
import torch.nn as nn
from torch.nn import functional as F
import torch

__all__ = ['FreqNet, freqnet']


def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = conv1x1(inplanes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class FreqNet(nn.Module):

    def __init__(self, block=Bottleneck, layers=[3, 4], num_classes=1, zero_init_residual=False):
        super(FreqNet, self).__init__()



        self.weight1 = nn.Parameter(torch.randn((64, 3, 1, 1)).cuda())
        self.bias1   = nn.Parameter(torch.randn((64,)).cuda())
        self.realconv1 = conv1x1(64, 64, stride=1)
        self.imagconv1 = conv1x1(64, 64, stride=1)

        self.weight2 = nn.Parameter(torch.randn((64, 64, 1, 1)).cuda())
        self.bias2   = nn.Parameter(torch.randn((64,)).cuda())
        self.realconv2 = conv1x1(64, 64, stride=1)
        self.imagconv2 = conv1x1(64, 64, stride=1)


        self.weight3 = nn.Parameter(torch.randn((256, 256, 1, 1)).cuda())
        self.bias3   = nn.Parameter(torch.randn((256,)).cuda())
        self.realconv3 = conv1x1(256, 256, stride=1)
        self.imagconv3 = conv1x1(256, 256, stride=1)

        self.weight4 = nn.Parameter(torch.randn((256, 256, 1, 1)).cuda())
        self.bias4   = nn.Parameter(torch.randn((256,)).cuda())
        self.realconv4 = conv1x1(256, 256, stride=1)
        self.imagconv4 = conv1x1(256, 256, stride=1)

        self.inplanes = 64
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])            # layer1 of resnet50
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2) # layer2 of resnet50
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 1)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def hfreqWH(self, x, scale):
        assert scale>2
        #print(f'input shape: {x.shape}, min: {x.min()}, max: {x.max()}, mean: {x.mean()}')
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        b,c,h,w = x.shape
        x[:,:,h//2-h//scale:h//2+h//scale,w//2-w//scale:w//2+w//scale ] = 0.0
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)
        #print(f'output shape: {x.shape}, min: {x.min()}, max: {x.max()}, mean: {x.mean()}')
        #print()
        return x

    def hfreqC(self, x, scale):
        assert scale>2
        # print(f'input shape: {x.shape}, min: {x.min()}, max: {x.max()}, mean: {x.mean()}')
        x = torch.fft.fft(x, dim=1, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=1)
        b,c,h,w = x.shape

        x[:,c//2-c//scale:c//2+c//scale,:,:] = 0.0

        x = torch.fft.ifftshift(x, dim=1)
        x = torch.fft.ifft(x, dim=1, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)

        # print(f'output shape: {x.shape}, min: {x.min()}, max: {x.max()}, mean: {x.mean()}')
        # print()
        return x


    def forward(self, x):

        ### HFRI
        x = self.hfreqWH(x, 4)
        x = F.conv2d(x, self.weight1, self.bias1, stride=1, padding=0)
        x = F.relu(x, inplace=True)

        ### HFRFC
        x = self.hfreqC(x, 4)

        ### FCL
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(self.realconv1(x.real), self.imagconv1(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)

        ### HFRFS
        x = self.hfreqWH(x, 4)
        x = F.conv2d(x, self.weight2, self.bias2, stride=2, padding=0)
        x = F.relu(x, inplace=True)

        ### HFRFC
        x = self.hfreqC(x, 4)

        ### FCL
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(self.realconv2(x.real), self.imagconv2(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)


        x = self.maxpool(x)
        x = self.layer1(x)# in64 out256


        ### HFRFS
        x = self.hfreqWH(x, 4)
        x = F.conv2d(x, self.weight3, self.bias3, stride=1, padding=0)
        x = F.relu(x, inplace=True)


        ### FCL
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(self.realconv3(x.real), self.imagconv3(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)

        ### HFRFS
        x = self.hfreqWH(x, 4)
        x = F.conv2d(x, self.weight4, self.bias4, stride=2, padding=0)
        x = F.relu(x, inplace=True)

        ### FCL
        x = torch.fft.fft2(x, norm="ortho")#,norm='forward'
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(self.realconv4(x.real), self.imagconv4(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho")
        x = torch.real(x)
        x = F.relu(x, inplace=True)

        x = self.layer2(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)

        return x

def freqnet(**kwargs):

    return FreqNet()



Writing freqnet1.py


In [ ]:
%%writefile base_model.py
import os
import torch
import torch.nn as nn
from torch.nn import init
from torch.optim import lr_scheduler


class BaseModel(nn.Module):
    def __init__(self, opt):
        super(BaseModel, self).__init__()
        self.opt = opt
        self.total_steps = 0
        self.isTrain = opt.isTrain
        self.lr = opt.lr
        self.save_dir = os.path.join(opt.checkpoints_dir, opt.name)
        #self.device = torch.device('cuda:{}'.format(opt.gpu_ids[0])) if opt.gpu_ids else torch.device('cpu')
        self.device =  torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def save_networks(self, epoch):
        save_filename = 'model_epoch_%s.pth' % epoch
        save_path = os.path.join(self.save_dir, save_filename)

        # serialize model and optimizer to dict
        # state_dict = {
        #     'model': self.model.state_dict(),
        #     'optimizer' : self.optimizer.state_dict(),
        #     'total_steps' : self.total_steps,
        # }

        torch.save(self.model.state_dict(), save_path)
        print(f'Saving model {save_path}')

    # load models from the disk
    def load_networks(self, epoch):
        load_filename = 'model_epoch_%s.pth' % epoch
        load_path = os.path.join(self.save_dir, load_filename)

        print('loading the model from %s' % load_path)
        # if you are using PyTorch newer than 0.4 (e.g., built from
        # GitHub source), you can remove str() on self.device
        state_dict = torch.load(load_path, map_location=self.device)
        if hasattr(state_dict, '_metadata'):
            del state_dict._metadata

        self.model.load_state_dict(state_dict['model'])
        self.total_steps = state_dict['total_steps']

        if self.isTrain and not self.opt.new_optim:
            self.optimizer.load_state_dict(state_dict['optimizer'])
            ### move optimizer state to GPU
            for state in self.optimizer.state.values():
                for k, v in state.items():
                    if torch.is_tensor(v):
                        state[k] = v.to(self.device)

            for g in self.optimizer.param_groups:
                g['lr'] = self.opt.lr

    def eval(self):
        self.model.eval()

    def train(self):
        self.model.train()

    def test(self):
        with torch.no_grad():
            self.forward()


def init_weights(net, init_type='normal', gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and (classname.find('Conv') != -1 or classname.find('Linear') != -1):
            if init_type == 'normal':
                init.normal_(m.weight.data, 0.0, gain)
            elif init_type == 'xavier':
                init.xavier_normal_(m.weight.data, gain=gain)
            elif init_type == 'kaiming':
                init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')
            elif init_type == 'orthogonal':
                init.orthogonal_(m.weight.data, gain=gain)
            else:
                raise NotImplementedError('initialization method [%s] is not implemented' % init_type)
            if hasattr(m, 'bias') and m.bias is not None:
                init.constant_(m.bias.data, 0.0)
        elif classname.find('BatchNorm2d') != -1:
            init.normal_(m.weight.data, 1.0, gain)
            init.constant_(m.bias.data, 0.0)

    print('initialize network with %s' % init_type)
    net.apply(init_func)

Writing base_model.py


In [ ]:
import urllib.request

path = "/content/4-classes-freqnet-v2.pth"

url = "https://raw.githubusercontent.com/chuangchuangtan/FreqNet-DeepfakeDetection/main/4-classes-freqnet-v2.pth"

urllib.request.urlretrieve(url, path)

print("Downloaded:", path)

Downloaded: /content/4-classes-freqnet-v2.pth


In [ ]:
import torch

checkpoint = torch.load(path, map_location="cpu", weights_only=True)

print("Type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("First keys:", list(checkpoint.keys())[:20])

Type: <class 'collections.OrderedDict'>
First keys: ['weight1', 'bias1', 'weight2', 'bias2', 'weight3', 'bias3', 'weight4', 'bias4', 'realconv1.weight', 'imagconv1.weight', 'realconv2.weight', 'imagconv2.weight', 'realconv3.weight', 'imagconv3.weight', 'realconv4.weight', 'imagconv4.weight', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean']


In [ ]:
%%writefile trainer.py
import functools
import torch
import torch.nn as nn
from freqnet import freqnet
from base_model import BaseModel, init_weights


class Trainer(BaseModel):
    def name(self):
        return 'Trainer'

    def __init__(self, opt):
        super(Trainer, self).__init__(opt)

        self.model = freqnet()

        for name,pa in self.model.named_parameters():
            if pa.requires_grad: print('='*20, 'requires_grad Ture',name)
        for name,pa in self.model.named_parameters():
            if not pa.requires_grad: print('='*20, 'requires_grad False',name)
        print()

        if self.isTrain:
            self.loss_fn = nn.BCEWithLogitsLoss()
            # initialize optimizers
            if opt.optim == 'adam':
                self.optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, self.model.parameters()),
                                                  lr=opt.lr, betas=(opt.beta1, 0.999))
            elif opt.optim == 'sgd':
                self.optimizer = torch.optim.SGD(filter(lambda p: p.requires_grad, self.model.parameters()),
                                                 lr=opt.lr, momentum=0.0, weight_decay=0)
            else:
                raise ValueError("optim should be [adam, sgd]")

        if not self.isTrain or opt.continue_train:
            self.load_networks(opt.epoch)
        self.model.to(opt.gpu_ids[0])


    def adjust_learning_rate(self, min_lr=1e-6):
        for param_group in self.optimizer.param_groups:
            param_group['lr'] *= 0.8
            if param_group['lr'] < min_lr:
                return False
        self.lr = param_group['lr']
        print('*'*25)
        print(f'Changing lr from {param_group["lr"]/0.8} to {param_group["lr"]}')
        print('*'*25)
        return True

    def set_input(self, input):
        self.input = input[0].to(self.device)
        self.label = input[1].to(self.device).float()


    def forward(self):
        self.output = self.model(self.input)

    def get_loss(self):
        return self.loss_fn(self.output.squeeze(1), self.label)

    def optimize_parameters(self):
        self.forward()
        self.loss = self.loss_fn(self.output.squeeze(1), self.label)
        self.optimizer.zero_grad()
        self.loss.backward()
        self.optimizer.step()


Writing trainer.py


In [ ]:
%%writefile test_options.py

# Paste your complete util.py code below
from base_options import BaseOptions


class TestOptions(BaseOptions):
    def initialize(self, parser):
        parser = BaseOptions.initialize(self, parser)
        # parser.add_argument('--dataroot')
        parser.add_argument('--model_path')
        parser.add_argument('--no_resize', action='store_true')
        parser.add_argument('--no_crop', action='store_true')
        parser.add_argument('--eval', action='store_true', help='use eval mode during test time.')
        parser.add_argument('--earlystop_epoch', type=int, default=15)
        parser.add_argument('--lr', type=float, default=0.00002, help='initial learning rate for adam')
        parser.add_argument('--niter', type=int, default=0, help='# of iter at starting learning rate')

        self.isTrain = False
        return parser

Writing test_options.py


In [ ]:
import shutil

free_gb = shutil.disk_usage("/content").free / (1024**3)
print("Free storage:", round(free_gb, 2), "GB")

Free storage: 112.44 GB


In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download
import os

zip_path = hf_hub_download(
    repo_id="sywang/CNNDetection",
    filename="progan_testset.zip",
    repo_type="dataset",
    local_dir="/content"
)

print("Downloaded:", zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / (1024**2), 2))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'TypeError: Failed to fetch'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


progan_testset.zip: reconstructing file:   0%|          |  0.00B /  834MB            

progan_testset.zip: downloading bytes:           |  0.00B            

Downloaded: /content/progan_testset.zip
Size (MB): 795.53


In [ ]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as z:
    print("\n".join(z.namelist()[:30]))

progan/
progan/airplane/
progan/airplane/0_real/
progan/airplane/0_real/11801.png
progan/airplane/0_real/05970.png
progan/airplane/0_real/12307.png
progan/airplane/0_real/03294.png
progan/airplane/0_real/17680.png
progan/airplane/0_real/18560.png
progan/airplane/0_real/14798.png
progan/airplane/0_real/10175.png
progan/airplane/0_real/10104.png
progan/airplane/0_real/19433.png
progan/airplane/0_real/16670.png
progan/airplane/0_real/14864.png
progan/airplane/0_real/15133.png
progan/airplane/0_real/13761.png
progan/airplane/0_real/08949.png
progan/airplane/0_real/03199.png
progan/airplane/0_real/03655.png
progan/airplane/0_real/03986.png
progan/airplane/0_real/09372.png
progan/airplane/0_real/17058.png
progan/airplane/0_real/00253.png
progan/airplane/0_real/19277.png
progan/airplane/0_real/06462.png
progan/airplane/0_real/11336.png
progan/airplane/0_real/07013.png
progan/airplane/0_real/06142.png
progan/airplane/0_real/12966.png


In [ ]:
import os

print("Current directory:", os.getcwd())
print("\nFiles in /content:")
print(os.listdir('/content'))

Current directory: /content

Files in /content:
['.config', 'train_options.py', '4-classes-freqnet-v2.pth', 'base_model.py', 'test_options.py', 'trainer.py', '.cache', 'progan_testset.zip', 'util.py', 'freqnet.py', 'datasets.py', 'base_options.py', 'freqnet1.py', 'init.py', 'sample_data']


In [ ]:
import zipfile
import os

zip_path = '/content/progan_testset.zip'
extract_path = '/content/dataset/ForenSynths'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

['progan']


In [ ]:
for root, dirs, files in os.walk(extract_path):
    print(root, "->", dirs[:5])
    if root.count(os.sep) - extract_path.count(os.sep) >= 2:
        dirs[:] = []

/content/dataset/ForenSynths -> ['progan']
/content/dataset/ForenSynths/progan -> ['train', 'horse', 'bird', 'chair', 'bus']
/content/dataset/ForenSynths/progan/train -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/horse -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/bird -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/chair -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/bus -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/person -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/diningtable -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/dog -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/bicycle -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/bottle -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/airplane -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/cow -> ['0_real', '1_fake']
/content/dataset/ForenSynths/progan/car -> ['0_real', '1_fake']
/content/data

In [ ]:
import os

print(os.listdir('/content/dataset'))
print(os.listdir('/content/dataset/ForenSynths'))

['ForenSynths']
['progan']


In [ ]:
%%writefile validate.py
import torch
import numpy as np
from freqnet import freqnet
from sklearn.metrics import average_precision_score, precision_recall_curve, accuracy_score
from test_options import TestOptions
from init import create_dataloader


def validate(model, opt):
    data_loader = create_dataloader(opt)

    with torch.no_grad():
        y_true, y_pred = [], []
        for img, label in data_loader:
            in_tens = img.cuda()
            y_pred.extend(model(in_tens).sigmoid().flatten().tolist())
            y_true.extend(label.flatten().tolist())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    r_acc = accuracy_score(y_true[y_true==0], y_pred[y_true==0] > 0.5)
    f_acc = accuracy_score(y_true[y_true==1], y_pred[y_true==1] > 0.5)
    acc = accuracy_score(y_true, y_pred > 0.5)
    ap = average_precision_score(y_true, y_pred)
    return acc, ap, r_acc, f_acc, y_true, y_pred


if __name__ == '__main__':
    opt = TestOptions().parse(print_options=False)

    model = freqnet(num_classes=1)
    state_dict = torch.load("/content/4-classes-freqnet-v2.pth", map_location='cpu')
    model.load_state_dict(state_dict, strict=True)
    model.cuda()
    model.eval()

    acc, avg_precision, r_acc, f_acc, y_true, y_pred = validate(model, opt)

    print("accuracy:", acc)
    print("average precision:", avg_precision)

    print("accuracy of real images:", r_acc)
    print("accuracy of fake images:", f_acc)

Writing validate.py


In [ ]:

import os
import time
import copy
import torch
import numpy as np

from validate import validate
from util import Logger, printSet
from test_options import TestOptions

# Import the two separate model implementations
from freqnet1 import freqnet as original_freqnet
from freqnet import freqnet as modified_freqnet


# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

CHECKPOINT_PATH = "/content/4-classes-freqnet-v2.pth"

DetectionTests = {
    "ForenSynths": {
        "dataroot": "/content/dataset/ForenSynths/",
        "no_resize": False,
        "no_crop": True,
    },
    "GANGen-Detection": {
        "dataroot": "/content/dataset/GANGen-Detection/",
        "no_resize": True,
        "no_crop": True,
    },
}


# --------------------------------------------------
# DEVICE AND CHECKPOINT
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Checkpoint:", CHECKPOINT_PATH)

if not os.path.isfile(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}"
    )

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

# Support either a direct state_dict or a wrapped checkpoint.
if isinstance(checkpoint, dict) and "model" in checkpoint:
    checkpoint = checkpoint["model"]

# Handle checkpoints saved with DataParallel's "module." prefix.
if checkpoint and all(
    key.startswith("module.") for key in checkpoint.keys()
):
    checkpoint = {
        key[len("module."):]: value
        for key, value in checkpoint.items()
    }


# --------------------------------------------------
# CREATE BOTH MODELS WITH THE SAME WEIGHTS
# --------------------------------------------------

models = {
    "Original FreqNet": original_freqnet(num_classes=1),
    "Modified FreqNet": modified_freqnet(num_classes=1),
}

for name, model in models.items():
    print(f"\nLoading weights into {name}...")

    # Strict loading catches architecture/key mismatches.
    model.load_state_dict(checkpoint, strict=True)
    model.to(device)
    model.eval()

    print(f"{name}: checkpoint loaded successfully.")


# --------------------------------------------------
# EVALUATION
# --------------------------------------------------

opt_base = TestOptions().parse(print_options=False)

all_results = {}

for testSet, config in DetectionTests.items():

    dataroot = config["dataroot"]

    # Skip datasets that have not been downloaded.
    if not os.path.isdir(dataroot):
        print(f"\nSkipping {testSet}: directory not found:")
        print(dataroot)
        continue

    # Only evaluate immediate subdirectories, as in your
    # original test.py.
    subfolders = sorted([
        folder for folder in os.listdir(dataroot)
        if os.path.isdir(os.path.join(dataroot, folder))
    ])

    if not subfolders:
        print(f"\nSkipping {testSet}: no subdirectories found.")
        continue

    print("\n" + "=" * 65)
    printSet(testSet)
    print("=" * 65)

    all_results[testSet] = {}

    for model_name, model in models.items():

        print(f"\nEvaluating: {model_name}")
        print(time.strftime("%Y-%m-%d %H:%M:%S"))

        accuracies = []
        average_precisions = []

        for index, folder in enumerate(subfolders):

            opt = copy.deepcopy(opt_base)

            opt.dataroot = os.path.join(dataroot, folder)
            opt.classes = ""
            opt.no_resize = config["no_resize"]
            opt.no_crop = config["no_crop"]

            result = validate(model, opt)

            acc, ap, *_ = result

            accuracies.append(float(acc))
            average_precisions.append(float(ap))

            print(
                f"({index} {folder:20}) "
                f"ACC: {acc * 100:.2f}% | "
                f"AP: {ap * 100:.2f}%"
            )

        mean_acc = float(np.mean(accuracies))
        mean_ap = float(np.mean(average_precisions))

        all_results[testSet][model_name] = {
            "accuracy": mean_acc,
            "ap": mean_ap,
            "per_folder_acc": accuracies,
            "per_folder_ap": average_precisions,
            "folders": subfolders,
        }

        print("-" * 55)
        print(f"{model_name} mean ACC: {mean_acc * 100:.2f}%")
        print(f"{model_name} mean AP : {mean_ap * 100:.2f}%")
        print("-" * 55)


# --------------------------------------------------
# FINAL COMPARISON
# --------------------------------------------------

print("\n\n" + "=" * 75)
print("ORIGINAL vs MODIFIED FREQNET — PERFORMANCE COMPARISON")
print("=" * 75)

for testSet, results in all_results.items():

    if (
        "Original FreqNet" not in results
        or "Modified FreqNet" not in results
    ):
        continue

    original = results["Original FreqNet"]
    modified = results["Modified FreqNet"]

    acc_diff = (
        modified["accuracy"] - original["accuracy"]
    ) * 100

    ap_diff = (
        modified["ap"] - original["ap"]
    ) * 100

    print(f"\nDataset: {testSet}")
    print(
        f"Original ACC : {original['accuracy'] * 100:.2f}%"
    )
    print(
        f"Modified ACC : {modified['accuracy'] * 100:.2f}%"
    )
    print(f"ACC change   : {acc_diff:+.2f} percentage points")

    print(
        f"Original AP  : {original['ap'] * 100:.2f}%"
    )
    print(
        f"Modified AP  : {modified['ap'] * 100:.2f}%"
    )
    print(f"AP change    : {ap_diff:+.2f} percentage points")

    if acc_diff > 0:
        print("Accuracy improved.")
    elif acc_diff < 0:
        print("Accuracy decreased.")
    else:
        print("Accuracy unchanged.")

    if ap_diff > 0:
        print("Average precision improved.")
    elif ap_diff < 0:
        print("Average precision decreased.")
    else:
        print("Average precision unchanged.")

print("\nEvaluation complete.")


Device: cuda
Checkpoint: /content/4-classes-freqnet-v2.pth

Loading weights into Original FreqNet...
Original FreqNet: checkpoint loaded successfully.

Loading weights into Modified FreqNet...
Modified FreqNet: checkpoint loaded successfully.

           ForenSynths

Evaluating: Original FreqNet
2026-11-06 14:00:41


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


(0 progan              ) ACC: 99.58% | AP: 100.00%
-------------------------------------------------------
Original FreqNet mean ACC: 99.58%
Original FreqNet mean AP : 100.00%
-------------------------------------------------------

Evaluating: Modified FreqNet
2026-11-06 14:04:00


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


(0 progan              ) ACC: 99.69% | AP: 100.00%
-------------------------------------------------------
Modified FreqNet mean ACC: 99.69%
Modified FreqNet mean AP : 100.00%
-------------------------------------------------------

Skipping GANGen-Detection: directory not found:
/content/dataset/GANGen-Detection/


ORIGINAL vs MODIFIED FREQNET — PERFORMANCE COMPARISON

Dataset: ForenSynths
Original ACC : 99.58%
Modified ACC : 99.69%
ACC change   : +0.11 percentage points
Original AP  : 100.00%
Modified AP  : 100.00%
AP change    : +0.00 percentage points
Accuracy improved.
Average precision improved.

Evaluation complete.
